En este notebook vamos a crear un clasificador Naive Bayes para resolver un problema de Sentiment Analysis de tweets. Este problema se enmarca dentro de NLU (Natural Language Understanding) ya que nos permitirá explotar los datos de tweets para extraer información de los mismos.

# 1. Importar bibliotecas necesarias

In [ ]:
import pandas as pd

# Bibliotecas para preprocesamiento y vectorización

from sklearn.feature_extraction.text import CountVectorizer
import nltk
from nltk.corpus import stopwords
import re

#Bibliotecas para entrenar el modelo
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 2. Analizamos el dataset

In [ ]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("ferno2/training1600000processednoemoticoncsv")
print("Path to dataset files:", path)

100%|██████████| 80.9M/80.9M [00:00<00:00, 195MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ferno2/training1600000processednoemoticoncsv/versions/1


In [ ]:
file_path = path + "/training.1600000.processed.noemoticon.csv"

df = pd.read_csv(
    file_path,
    encoding="latin-1",
    header=None,
    names=["target", "id", "date", "flag", "user", "text"],
    sep=",",             # correct separator
    quotechar='"',       # handles commas inside tweets
    engine="python",      # more flexible for irregular lines
    skiprows=1
)

print(df.head())
print(df.shape)

   target          id                          date      flag           user  \
0       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY  scotthamilton   
1       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY       mattycus   
2       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY        ElleCTF   
3       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY         Karoli   
4       0  1467811372  Mon Apr 06 22:20:00 PDT 2009  NO_QUERY       joy_wolf   

                                                text  
0  is upset that he can't update his Facebook by ...  
1  @Kenichan I dived many times for the ball. Man...  
2    my whole body feels itchy and like its on fire   
3  @nationwideclass no, it's not behaving at all....  
4                      @Kwesidei not the whole crew   
(1599999, 6)


In [ ]:
df.tail()

,target,id,date,flag,user,text
1297879,4,2193601966,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,AmandaMarie1028,Just woke up. Having no school is the best fee...
1297880,4,2193601969,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,TheWDBoards,TheWDB.com - Very cool to hear old Walt interv...
1297881,4,2193601991,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,bpbabe,Are you ready for your MoJo Makeover? Ask me f...
1297882,4,2193602064,Tue Jun 16 08:40:49 PDT 2009,NO_QUERY,tinydiamondz,Happy 38th Birthday to my boo of alll time!!! ...
1297883,4,2193602129,Tue Jun 16 08:40:50 PDT 2009,NO_QUERY,RyanTrevMorris,happy #charitytuesday @theNSPCC @SparksCharity...


In [ ]:
df['target'].value_counts()

,count
target,
4,800000
0,497884


**¿Es estructurado, semi-estructurado o no estructurado?**

**¿Cuantas observaciones tengo?**

**¿Qué variables voy a usar para entrenar el modelo de sentiment analysis?**

**¿En qué idioma está el texto? ¿Que limpieza y preprocesamiento será necesario llevar a cabo?**

**¿Qué valores toma la variable de salida? ¿Qué significa cada valor? ¿Las clases están balanceadas?**

# 3. Limpieza y Preprocesamiento

a. Descargamos stopwords en inglés

In [ ]:
nltk.download('stopwords')
stop_words = list(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


b. Definimos una función para limpiar el texto. Quitamos términos usando regex (regular expression)

In [ ]:
def clean_text(text):
    if text is None:  # Handle None values
        return ""
    text = re.sub(r'@\w+', '', text)  # Eliminar menciones de usuarios
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Eliminar URLs
    text = re.sub(r'\d+', '', text)  # Eliminar números
    text = re.sub(r'[^\w\s]', '', text)  # Eliminar caracteres especiales y puntuación
    text = re.sub(r'\b\w*[ñð¾ºµ¼]\w*\b', '', text)  # Eliminar palabras que contienen los caracteres especiales
    text = text.lower()  # Convertir a minúsculas
    return text

3. Nos quedamos solo con las columnas necesarias y limpiamos el texto

In [ ]:
df = df[['target', 'text']]
df['target'] = df['target'].map({0: 'negativo', 4: 'positivo'})

df['text'] = df['text'].apply(clean_text)

4. Hacemos una muestra para que sea más eficiente el entrenamiento

In [ ]:
df_sample = df.sample(n=50000, random_state=42)

df_sample

,target,text
541200,negativo,my poor little dumpling in holmdel vids he w...
750,negativo,im off too bed i gotta wake up hella early tom...
766711,negativo,i havent been able to listen to it yet my spe...
285055,negativo,now remembers why solving a relatively big equ...
705995,negativo,ate too much feel sick
...,...,...
199266,negativo,as much as i wanna eat this ham sandwhich i ca...
210814,negativo,ok i guess i will stop bsing and get on the hi...
180674,negativo,was planning on next month but apparently the...
364859,negativo,at work till tonight


**Chequear que las clases estén balanceadas**

5. Separamos X e y

In [ ]:
texts = df_sample['text'].tolist()
labels = df_sample['target'].tolist()

5. Vectorizamos textos y quitamos stopwords (se hace todo en un comando)

In [ ]:
?CountVectorizer

**¿Qué hace CountVectorizer?**

In [ ]:
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(texts)
X_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

X_df


,__,___,____,_____,______,_________,_____________,________________________________________________,_____s,___quotim,...,ñðñ,ñðññðññ,ñðñññ,ññ,ññððñ,ñññ,ññññ,ø³øøùø,ùø²ùù,ùø¹ùøù
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
49996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
49997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
49998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**¿Cuántas variables tiene el dataset? ¿Ya está listo para entrenar el modelo?**

# 4. Entrenamiento del Modelo

Ya tenemos los datos separados en X e y (se llaman 'X' y 'labels')

**En 5 líneas de código, entrenar un modelo de Naive Bayes (MultinomialNB()) y calcular accuracy en test**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.25, random_state=42)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Precisión del modelo: {accuracy:.2f}")

Precisión del modelo: 0.74


# 5. Análisis de Resultados

**¿Están conformes con la performance del modelo? ¿Qué se puede hacer para mejorarla?**


**Analicen las palabras más frecuentes dentro de cada clase ¿Tienen sentido?**


**¿Ven alguna manera de mejorar el preprocesamiento?**

In [ ]:
# Create sample tweets
sample_tweets = [
    "This was an ok flight.", # Negative
    "I dont think I could have a more pleasant experience with Data Airlines.",    # Negative
    "I need to have a conversation with a real person, I hate talking to chats" # Positive
]

# Clean and vectorize the sample tweets
cleaned_sample_tweets = [clean_text(tweet) for tweet in sample_tweets]
vectorized_sample_tweets = vectorizer.transform(cleaned_sample_tweets)

# Predict sentiment
predictions = model.predict(vectorized_sample_tweets)

# Display the predictions
for tweet, prediction in zip(sample_tweets, predictions):
    print(f"Tweet: '{tweet}' -> Predicted Sentiment: {prediction}")

Tweet: 'This was an ok flight.' -> Predicted Sentiment: negativo
Tweet: 'I dont think I could have a more pleasant experience with Data Airlines.' -> Predicted Sentiment: positivo
Tweet: 'I need to have a conversation with a real person, I hate talking to chats' -> Predicted Sentiment: positivo
